<a href="https://colab.research.google.com/github/ProfAndersonVanin/IBM3130-PLN-2026/blob/main/semana-03/Aula03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Aula 03 - Fundamentos Linguísticos — POS Tagging**


## **Etapa 1 - Instalação das Bibliotecas e Importação dos pacotes**

In [ ]:
# Instalação do spaCy
!pip install -q spacy

# Download do modelo em português
!python -m spacy download pt_core_news_sm

In [ ]:
import nltk
nltk.download('averaged_perceptron_tagger_eng', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

In [ ]:
import spacy
from spacy import displacy
from nltk.tokenize import word_tokenize
from nltk import pos_tag
from collections import Counter
import matplotlib.pyplot as plt

Carregar o modelo de processamento de linguagem natural em português de menor tamanho (sm ou small) da biblioteca spaCy, salvando-o na variável `nlp` para realizar análises de texto.

In [ ]:
nlp = spacy.load('pt_core_news_sm')

In [ ]:
nlp.pipe_names

O comando `nlp.pipe_names` retorna uma lista com os nomes de todos os componentes que estão ativos no fluxo de processamento (pipeline) do modelo spaCy. Quando você passa um texto para o modelo, ele passa por uma "*esteira de produção*" onde cada componente faz uma análise específica em sequência.

O que cada componente faz:
* `tok2vec`: Transforma as palavras em vetores numéricos para que o modelo entenda o contexto.
* `morphologizer`: Identifica as classes gramaticais (substantivo, verbo, adjetivo) e características morfológicas (gênero, número).
* `parser`: Descobre a estrutura sintática da frase (quem é o sujeito, qual é o objeto).
* `lemmatizer`: Encontra a forma base (infinitivo/singular) de cada palavra. Exemplo: "correndo" vira "correr".
* `attribute_ruler`: Aplica regras de correção para ajustar etiquetas gramaticais e lemas.
* `ner`: Reconhece entidades nomeadas no texto, como nomes de pessoas, locais, organizações e datas.

## **Etapa 2: POS Tagging com NLTK (INGLÊS)**

In [ ]:
frase_en = 'The neural network quickly processed the large dataset.'
tokens_en = word_tokenize(frase_en)
pos_en    = pos_tag(tokens_en)

In [ ]:
tokens_en

In [ ]:
pos_en

* DT: **Determinante** (Determiner) — **Artigos e demonstrativos** (ex: o, a, este).
* JJ: **Adjetivo** (Adjective) — Palavra que caracteriza o substantivo.
* NN: **Substantivo** singular (Noun, singular or mass) — Nome de coisas, objetos ou conceitos.
* RB: **Advérbio** (Adverb) — Palavra que modifica um verbo, adjetivo ou outro advérbio.
* VBD: **Verbo no passado** (Verb, past tense) — Ação que já aconteceu.
* .: **Pontuação** (Punctuation) — Sinais de fim de frase ou pausas.


Mais marcadores em: https://www.ling.upenn.edu/courses/Fall_2003/ling001/penn_treebank_pos.html

In [ ]:
desc = {'DT':'Determinante','JJ':'Adjetivo','NN':'Subs. singular',
         'NNS':'Subs. plural','VBD':'Verbo passado','VBN':'Particípio',
         'RB':'Advérbio','IN':'Preposição','CC':'Conj. coord.'}

print(f'{"Token":<18} {"Tag PTB":<10} {"Significado"}')
print('-' * 55)
for token, tag in pos_en:
    d = desc.get(tag, tag)
    print(f'{token:<18} {tag:<10} {d}')

## **Etapa 3: Ambiguidade contextual**

In [ ]:
# A mesma palavra com POS diferente dependendo do contexto
frases_amb = [
    'The model achieved high accuracy.',
    'We need to model the distribution.',
]


In [ ]:
for i, frase in enumerate(frases_amb):
    tokens = word_tokenize(frase)
    tags = pos_tag(tokens)
    print(i, " - Tokens : ", tokens)
    print(i, " - Tags : ", tags)
    print("-----------------")

    for tok, tag in tags:
        if tok.lower() == "model":
            print(frase)
            print("model =", tag)
            print()

## **Etapa 4: POS Tagging com spaCy em Português**

**Análise morfossintática completa**

In [ ]:
texto_pt = ('A pesquisadora brasileira desenvolveu um modelo de inteligência artificial, que analisa sentimentos em textos médicos com alta precisão.')

In [ ]:
doc = nlp(texto_pt)

In [ ]:
doc

**Objeto** `doc`: É um objeto especial do spaCy. **Ele não é uma string de texto simples**; é uma estrutura rica de dados que contém o texto original mapeado com todas as análises feitas pela inteligência artificial.

Após rodar essa linha, você pode acessar facilmente as informações extraídas do texto:
* **Iteração por palavras**: doc se comporta como uma lista de palavras (tokens).
* **Acesso às propriedades**: Cada palavra dentro de doc possui atributos automáticos como `.pos_` (classe gramatical), `.lemma_` (forma base da palavra) e `.dep_` (dependência sintática).
* **Entidades**: Você pode acessar as entidades nomeadas (nomes, locais) encontradas no texto através de `doc.ents`.

In [ ]:
for token in doc:
    print(f"Palavra: {token.text} -> Classe Gramatical: {token.pos_} -> Dependência Sintática: {token.dep_}")

In [ ]:
for token in doc:
    print(f"Stop Word? : {token.is_stop}")

In [ ]:
for token in doc:
    print(f"Lemma : {token.lemma_}")

In [ ]:
for token in doc:
    print(f"Palavra: {token.text} -> {token.tag_}")

## **Etapa 5: Filtrar tokens por classe gramatical**

In [ ]:
corpus = ['A inteligência artificial transforma a medicina moderna.',
          'Pesquisadores brasileiros desenvolvem modelos inovadores de PLN.',
          'O Brasil cresce no ranking de inovação tecnológica.']

doc_corpus = nlp(' '.join(corpus))

In [ ]:
doc_corpus

In [ ]:
def extrair_por_pos(doc, classes):
    resultado = []

    for token in doc:
        if token.pos_ in classes:
            if token.is_alpha and not token.is_stop:
                resultado.append(token.lemma_.lower())

    return resultado

In [ ]:
print('SUBSTANTIVOS:', extrair_por_pos(doc_corpus, ['NOUN','PROPN']))
print('VERBOS:      ', extrair_por_pos(doc_corpus, ['VERB']))
print('ADJETIVOS:   ', extrair_por_pos(doc_corpus, ['ADJ']))

# **Visualização com displacy**

## **Etapa 6: displacy modo 'dep' — árvore de dependências**

In [ ]:
frase_dep = 'O modelo neural classificou os textos rapidamente.'
doc_dep = nlp(frase_dep)

In [ ]:
for token in doc_dep:
    if not token.is_punct:
        print(f'{token.text:<18}{token.pos_:<8}{token.dep_:<12}{token.head.text}')


In [ ]:
# Renderizar a árvore no Colab
displacy.render(doc_dep, style='dep', jupyter=True, options={'distance':100,'compact':True})


## **Etapa 7: displacy personalizado para POS**

In [ ]:
# Visualizar POS tags com cores usando displacy no modo 'ent'

# CORES PARA CADA TIPO DE TAG
cores = {'NOUN':'#2E75B6','PROPN':'#1F4E79','VERB':'#1A5E3A',
         'AUX':'#0F6E56','ADJ':'#C45911','ADV':'#7B61C8',
         'ADP':'#888888','DET':'#AAAAAA'}


In [ ]:
frase_vis = 'O pesquisador desenvolveu um modelo neural muito eficiente.'
doc_vis = nlp(frase_vis)

In [ ]:
ents = []
for token in doc_vis:
    if not token.is_space:
        ents.append({'start':token.idx,'end':token.idx+len(token.text),'label':token.pos_})


In [ ]:
ents

In [ ]:
displacy.render({'text':frase_vis,'ents':ents,'title':None},
                style='ent',manual=True,jupyter=True,
                options={'ents':list(cores.keys()),'colors':cores})


## **Etapa 8: spaCy Matcher — Padrões de POS**

**Extraindo sintagmas nominais com padrões**

In [ ]:
nlp

In [ ]:
from spacy.matcher import Matcher
matcher = Matcher(nlp.vocab)


**matcher** ==> cria uma instância do Matcher, que é a ferramenta do spaCy usada para encontrar padrões de texto baseados em regras (uma espécie de "Expressões Regulares / Regex" turbinada, que entende gramática).

In [ ]:
# Padrão: [ADJ?] NOUN [ADJ*]
padrao_np = [{'POS':'ADJ','OP':'?'},
             {'POS':'NOUN'},
             {'POS':'ADJ','OP':'*'}]
matcher.add('SINTAGMA_NOMINAL', [padrao_np])


Este trecho de código define e registra uma regra no Matcher para encontrar Sintagmas Nominais (em inglês, Noun Phrases). Um sintagma nominal é um grupo de palavras centrado em um substantivo.

A regra busca por combinações específicas de Substantivos (`NOUN`) e Adjetivos (`ADJ`) usando operadores de repetição (`OP`), funcionando de forma muito parecida com expressões regulares (*Regex*), mas operando sobre classes gramaticais.

In [ ]:
# Padrão: NOUN + ADP + NOUN
padrao_comp = [{'POS':'NOUN'},
               {'POS':'ADP'},
               {'POS':{'IN':['NOUN','PROPN']}}]
matcher.add('COMPOSTO_NOMINAL', [padrao_comp])


In [ ]:
texto_ext = 'O sistema de processamento de linguagem natural identifica entidades.'
doc_ext = nlp(texto_ext)


Agora `doc_ext` contém os tokens analisados.

Podemos imaginar algo próximo de:

```
O          DET
sistema    NOUN
de         ADP
processamento NOUN
de         ADP
linguagem  NOUN
natural    ADJ
identifica VERB
entidades  NOUN
```



In [ ]:
matches = matcher(doc_ext)

Cada resultado contém três informações principais: `match_id, start, end`

In [ ]:
matches

In [ ]:
vistos = set()

Um set é uma estrutura que não permite elementos repetidos.

In [ ]:
for match_id, start, end in matches:
    span = doc_ext[start:end]
    nome = nlp.vocab.strings[match_id]
    if span.text not in vistos:
        vistos.add(span.text)
        tags = [f'{t.text}/{t.pos_}' for t in span]
        print(f'[{nome}] "{span.text}" — {tags}')


# **Mini-ABSA com POS e Dependências**

## **Etapa 9: Sentimento por aspecto com `token.dep_`**

In [ ]:
positivos = {'excelente','ótimo','incrível','rápido','preciso','eficiente','bom'}
negativos  = {'ruim','péssimo','horrível','lento','fraco','quebrado','caro'}

In [ ]:
reviews = [
    'A câmera é excelente mas a bateria é fraca.',
    'O design é elegante e o processador é muito rápido.',
    'O preço é caro demais para um produto tão fraco.',
]

In [ ]:
for review in reviews:
    doc_r = nlp(review)
    print(f'Review: "{review}"')
    for token in doc_r:
        if token.pos_ == 'ADJ' and token.head.pos_ in ['NOUN','PROPN']:
            aspecto = token.head.lemma_
            adj     = token.lemma_.lower()
            if adj in positivos:
                print(f'  ✅ {aspecto} → {adj} (POSITIVO)')
            elif adj in negativos:
                print(f'  ❌ {aspecto} → {adj} (NEGATIVO)')
    print()


In [ ]:
for review in reviews:
    doc_r = nlp(review)

    print(f'Review: "{review}"')

    for i, token in enumerate(doc_r):
        if token.lemma_.lower() in positivos:
            print("✅", doc_r[i-1].lemma_, "→", token.lemma_, "(POSITIVO)")

        elif token.lemma_.lower() in negativos:
            print("❌", doc_r[i-1].lemma_, "→", token.lemma_, "(NEGATIVO)")

    print()